# 사후설문 이원분산분석 가정 검정

이 노트북은 04번 이원분산분석에 사용한 7개 사후설문 척도에 대해 다음 가정을 점검합니다.

1. **잔차 정규성:** Shapiro–Wilk 검정과 Q–Q plot
2. **집단 간 등분산성:** 중앙값 기준 Levene 검정과 집단별 분포 그림

각 단계는 설명, 짧은 코드, 결과 확인, 해석 순서로 구성했습니다. 위에서 아래로 차례대로 실행하면 됩니다.


## 0. 검정 기준과 해석 방법

| 가정 | 검정 대상 | 귀무가설 | 판단 기준 |
|---|---|---|---|
| 정규성 | 각 종속변수의 ANOVA 모형 잔차 | 잔차가 정규분포를 따른다 | p<.05이면 점검 필요 |
| 등분산성 | UI×숙련도 4집단 | 네 집단의 분산이 같다 | p<.05이면 점검 필요 |

정규성은 설문 원점수 전체가 아니라 **각 이원분산분석 모형의 잔차**를 대상으로 확인합니다. 검정의 p값만으로 결론 내리지 않고 Q–Q plot과 집단별 분포도 함께 봅니다.


## 1. 분석 환경과 원자료 준비

### 1-1. 라이브러리 불러오기

데이터 처리에는 pandas, 검정에는 scipy와 statsmodels, 시각화에는 matplotlib과 seaborn을 사용합니다.


In [ ]:
from pathlib import Path
import sys

A2UI_DIR = Path.cwd() if (Path.cwd() / "analysis_common.py").exists() else Path.cwd() / "Data" / "A2UI"
if str(A2UI_DIR) not in sys.path:
    sys.path.insert(0, str(A2UI_DIR))
import analysis_common as common
from functools import partial
from analysis_common import significance_stars, format_p

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from IPython.display import Image, display


### 1-2. 파일과 그림 저장 위치 정하기

프로젝트 루트와 `Data/A2UI` 폴더 중 어디에서 실행해도 같은 원자료를 찾도록 설정합니다.


In [ ]:
A2UI_DIR = common.A2UI_DIR
SURVEY_FILE = common.SURVEY_FILE
FIGURE_DIR = common.FIGURE_DIR
KOREAN_FONT = common.configure_plots()


### 1-3. 설문 원자료 읽기

04번 노트북과 같은 `병합 데이터` 시트를 사용합니다.


In [ ]:
survey = common.load_survey()

print(f"자료 크기: {survey.shape[0]}명 × {survey.shape[1]}개 열")
display(survey[["참여자번호", "인터페이스"]].head())


## 2. 7개 사후설문 척도와 분석 집단 만들기

### 2-1. 척도별 사용 문항 정의하기

04번 이원분산분석과 동일한 문항 구성을 사용합니다.


In [ ]:
DV_ITEMS = common.DV_ITEMS
LITERACY_ITEMS = common.LITERACY_ITEMS


In [ ]:
item_map = pd.DataFrame({
    "종속변수": list(DV_ITEMS),
    "사용 문항": [
        ", ".join(items) for items in DV_ITEMS.values()
    ],
})
display(item_map)


### 2-2. 문항 번호로 실제 엑셀 열 찾기

엑셀 열 이름은 문항 번호와 질문 문장을 함께 포함합니다. `2-1.`처럼 시작하는 열을 정확히 하나씩 찾습니다.


In [ ]:
find_columns = partial(common.find_columns, survey)
average_score = partial(common.average_score, survey)


### 2-4. 7개 종속변수 계산하기

한 행은 참가자 한 명이고, 일곱 열은 04번 ANOVA에 사용된 사후설문 척도입니다.


In [ ]:
scores = common.make_scores(survey)

assert scores.notna().all().all()
print(
    f"점수표: {scores.shape[0]}명 × "
    f"{scores.shape[1]}개 종속변수"
)
display(scores.head().round(2))


### 2-5. UI와 숙련도 만들기

숙련도는 사전 AI 리터러시 6개 문항의 평균이 6.0 이상이면 숙련자, 미만이면 초보자로 구분합니다.


In [ ]:
design = common.make_design(survey)
literacy_pre = design["literacy_pre"]



### 2-6. 분석용 데이터와 네 집단의 크기 확인하기

가정 검정에서는 A2UI-초보자, A2UI-숙련자, Baseline-초보자, Baseline-숙련자의 네 집단을 비교합니다.


In [ ]:
analysis_data = pd.concat([design, scores], axis=1)
analysis_data["집단"] = (
    analysis_data["UI"] + " · " + analysis_data["숙련도"]
)

GROUP_ORDER = [
    "A2UI · 초보자", "A2UI · 숙련자",
    "Baseline · 초보자", "Baseline · 숙련자",
]
group_counts = analysis_data["집단"].value_counts().reindex(GROUP_ORDER)
display(group_counts.rename("n").to_frame())


**확인:** 최신 자료의 네 집단 표본 수는 각각 10명입니다. 표본 수가 작기 때문에 p값만 보지 않고 Q–Q plot과 분포 그림도 함께 확인해야 합니다.


## 3. 이원분산분석 모형과 잔차 만들기

### 3-1. 종속변수별 모형 적합 함수

각 척도에 `UI × 숙련도` 이원분산분석 모형을 적합합니다. 정규성 검정에는 이 모형에서 나온 잔차를 사용합니다.


In [ ]:
def fit_anova_model(variable):
    return common.fit_anova(design, scores[variable])[0]


### 3-2. 일곱 모형 적합하기

각 종속변수 이름을 키로 사용해 모형을 보관합니다.


In [ ]:
models = {
    variable: fit_anova_model(variable)
    for variable in DV_ITEMS
}

model_check = pd.DataFrame({
    "종속변수": list(models),
    "N": [int(model.nobs) for model in models.values()],
    "잔차 자유도": [int(model.df_resid) for model in models.values()],
})
display(model_check.set_index("종속변수"))


## 4. 잔차 정규성 검정

### 4-1. Shapiro–Wilk 검정

귀무가설은 “모형 잔차가 정규분포를 따른다”입니다. p<.05이면 정규성 위반 가능성을 점검합니다.


In [ ]:
normality_rows = []
for variable, model in models.items():
    test = stats.shapiro(model.resid)
    normality_rows.append({
        "종속변수": variable,
        "Shapiro W": test.statistic,
        "p": test.pvalue,
        "판단": (
            "위반 근거 없음"
            if test.pvalue >= .05 else "점검 필요"
        ),
    })
normality_results = pd.DataFrame(normality_rows)


In [ ]:
normality_display = normality_results.copy()
normality_display["Shapiro W"] = normality_display["Shapiro W"].round(3)
normality_display["p"] = normality_display["p"].round(3)
display(normality_display.set_index("종속변수"))


### 4-2. Q–Q plot 함수 만들기

점들이 기준선에 가깝게 놓이면 정규분포와 대체로 일치합니다. 양끝에서 크게 벗어나는 점은 꼬리 분포나 이상치를 확인해야 한다는 신호입니다.


In [ ]:
PALETTE = {
    "ok": "#2A6F97",
    "check": "#D1495B",
    "line": "#59636E",
    "grid": "#D9DEE3",
}

def draw_qq_plot(ax, variable, model, p_value):
    theoretical, ordered = stats.probplot(
        model.resid, dist="norm", fit=False
    )
    slope, intercept, _ = stats.probplot(
        model.resid, dist="norm", fit=True
    )[1]
    color = PALETTE["ok"] if p_value >= .05 else PALETTE["check"]
    ax.scatter(theoretical, ordered, s=28, color=color, alpha=.85)
    ax.plot(theoretical, slope * theoretical + intercept,
            color=PALETTE["line"], linewidth=1.4)
    ax.set_title(f"{variable}\nShapiro p={p_value:.3f}", fontsize=10)


### 4-3. 일곱 척도의 Q–Q plot 그리기

빨간 점은 Shapiro–Wilk p<.05인 척도이고, 파란 점은 p≥.05인 척도입니다.


In [ ]:
qq_png = FIGURE_DIR / "figure_04_1_residual_qqplots.png"
qq_pdf = FIGURE_DIR / "figure_04_1_residual_qqplots.pdf"

with plt.rc_context({"font.family": KOREAN_FONT, "axes.unicode_minus": False}):
    fig, axes = plt.subplots(2, 4, figsize=(15, 7.6))
    for ax, variable in zip(axes.flat, DV_ITEMS):
        p_value = normality_results.loc[
            normality_results["종속변수"].eq(variable), "p"
        ].iloc[0]
        draw_qq_plot(ax, variable, models[variable], p_value)
        ax.grid(color=PALETTE["grid"], linewidth=.7, alpha=.7)
        ax.set_xlabel("이론적 분위수")
        ax.set_ylabel("잔차 분위수")
    axes.flat[-1].axis("off")
    fig.suptitle("사후설문 ANOVA 잔차 Q–Q plot", fontsize=15, y=1.02)
    fig.tight_layout()
    fig.savefig(qq_png, dpi=300, bbox_inches="tight")
    fig.savefig(qq_pdf, bbox_inches="tight")
    plt.close(fig)
display(Image(filename=qq_png))


**정규성 결과 확인:** 위 Shapiro–Wilk 결과표에서 p<.05인 척도를 확인하고, Q–Q plot을 함께 검토합니다. 수치는 현재 입력 자료에서 계산한 표를 기준으로 합니다.

---

## 5. 집단 간 등분산성 검정

### 5-1. Levene 검정을 위한 네 집단 값 만들기

중앙값을 중심으로 계산하는 Levene 검정은 Brown–Forsythe 방식으로도 불리며, 평균 중심 검정보다 비정규성에 덜 민감합니다.


In [ ]:
def values_by_group(variable):
    return [
        analysis_data.loc[
            analysis_data["집단"].eq(group), variable
        ].to_numpy()
        for group in GROUP_ORDER
    ]


### 5-2. 일곱 척도의 Levene 검정 실행하기

귀무가설은 “네 집단의 분산이 같다”입니다. p<.05이면 등분산성 위반 가능성을 점검합니다.


In [ ]:
variance_rows = []
for variable in DV_ITEMS:
    test = stats.levene(
        *values_by_group(variable),
        center="median",
    )
    variance_rows.append({
        "종속변수": variable,
        "Levene F": test.statistic,
        "df1": len(GROUP_ORDER) - 1,
        "df2": len(analysis_data) - len(GROUP_ORDER),
        "p": test.pvalue,
        "판단": (
            "위반 근거 없음"
            if test.pvalue >= .05 else "점검 필요"
        ),
    })
variance_results = pd.DataFrame(variance_rows)


In [ ]:
variance_display = variance_results.copy()
variance_display["Levene F"] = variance_display["Levene F"].round(3)
variance_display["p"] = variance_display["p"].round(3)
display(variance_display.set_index("종속변수"))


### 5-3. 집단별 분포 그림 함수 만들기

상자그림은 중앙값과 사분위 범위를 보여주고, 점은 참가자의 실제 척도 점수를 나타냅니다.


In [ ]:
GROUP_COLORS = ["#5B8FF9", "#61DDAA", "#F6BD16", "#E8684A"]

def draw_group_distribution(ax, variable):
    sns.boxplot(
        data=analysis_data, x="집단", y=variable, hue="집단",
        order=GROUP_ORDER, palette=GROUP_COLORS,
        width=.58, showfliers=False, legend=False, ax=ax,
    )
    sns.stripplot(
        data=analysis_data, x="집단", y=variable,
        order=GROUP_ORDER, color="#263238",
        size=3.8, jitter=.15, alpha=.72, ax=ax,
    )
    p_value = variance_results.loc[
        variance_results["종속변수"].eq(variable), "p"
    ].iloc[0]
    ax.set_title(f"{variable}\nLevene p={p_value:.3f}", fontsize=10)
    ax.set_xlabel("")
    ax.set_ylabel("척도 점수")


### 5-4. 일곱 척도의 4집단 분포 비교하기

각 패널에서 상자의 높이와 점의 퍼짐이 집단마다 크게 다른지 확인합니다.


In [ ]:
box_png = FIGURE_DIR / "figure_04_1_group_distributions.png"
box_pdf = FIGURE_DIR / "figure_04_1_group_distributions.pdf"

with plt.rc_context({"font.family": KOREAN_FONT, "axes.unicode_minus": False}):
    fig, axes = plt.subplots(2, 4, figsize=(17, 8.2))
    for ax, variable in zip(axes.flat, DV_ITEMS):
        draw_group_distribution(ax, variable)
        ax.tick_params(axis="x", rotation=25, labelsize=8)
        ax.grid(axis="y", color=PALETTE["grid"], linewidth=.7, alpha=.7)
    axes.flat[-1].axis("off")
    fig.suptitle("사후설문 UI×숙련도 집단별 분포", fontsize=15, y=1.02)
    fig.tight_layout()
    fig.savefig(box_png, dpi=300, bbox_inches="tight")
    fig.savefig(box_pdf, bbox_inches="tight")
    plt.close(fig)
display(Image(filename=box_png))


**등분산성 결과 확인:** 위 Levene 결과표에서 p<.05인 척도를 확인합니다. 정규성과 등분산성을 함께 고려한 변환 재분석은 04-2번에서 수행합니다.

## 6. 가정 검정 결과를 한 표로 정리하기

### 6-1. 정규성과 등분산성 결과 결합하기

두 검정 중 하나라도 p<.05이면 `점검 필요`로 표시합니다.


In [ ]:
assumption_summary = normality_results[
    ["종속변수", "Shapiro W", "p"]
].rename(columns={"p": "정규성 p"})
assumption_summary = assumption_summary.merge(
    variance_results[["종속변수", "Levene F", "p"]]
    .rename(columns={"p": "등분산성 p"}),
    on="종속변수",
)
assumption_summary["종합 판단"] = np.where(
    (assumption_summary["정규성 p"] >= .05)
    & (assumption_summary["등분산성 p"] >= .05),
    "위반 근거 없음", "점검 필요",
)


In [ ]:
summary_display = assumption_summary.copy()
for column in ["Shapiro W", "정규성 p", "Levene F", "등분산성 p"]:
    summary_display[column] = summary_display[column].round(3)
display(summary_display.set_index("종속변수"))


### 6-2. 두 검정의 p값을 한 그림에서 비교하기

점선은 유의수준 .05입니다. 점이 점선보다 왼쪽에 있으면 해당 가정을 추가로 점검해야 합니다.


In [ ]:
plot_data = assumption_summary.melt(
    id_vars="종속변수",
    value_vars=["정규성 p", "등분산성 p"],
    var_name="검정", value_name="p",
)
plot_data["종속변수"] = pd.Categorical(
    plot_data["종속변수"],
    categories=list(reversed(DV_ITEMS)),
    ordered=True,
)


In [ ]:
summary_png = FIGURE_DIR / "figure_04_1_assumption_pvalues.png"
summary_pdf = FIGURE_DIR / "figure_04_1_assumption_pvalues.pdf"

with plt.rc_context({"font.family": KOREAN_FONT, "axes.unicode_minus": False}):
    fig, ax = plt.subplots(figsize=(9.4, 5.7))
    sns.scatterplot(
        data=plot_data, x="p", y="종속변수", hue="검정",
        style="검정", s=95,
        palette={"정규성 p": "#2A6F97", "등분산성 p": "#E07A5F"},
        ax=ax,
    )
    ax.axvline(.05, color="#C1121F", linestyle="--", linewidth=1.4,
               label="유의수준 .05")
    ax.set_xlim(left=0)
    ax.set_xlabel("p값")
    ax.set_ylabel("")
    ax.set_title("사후설문 ANOVA 가정 검정 p값")
    ax.grid(axis="x", color=PALETTE["grid"], linewidth=.7, alpha=.75)
    ax.legend(frameon=False, loc="lower right")
    fig.tight_layout()
    fig.savefig(summary_png, dpi=300, bbox_inches="tight")
    fig.savefig(summary_pdf, bbox_inches="tight")
    plt.close(fig)
display(Image(filename=summary_png))


## 7. 최종 해석

최신 자료의 가정 검정 결과는 위의 Shapiro–Wilk·Levene 결과표와 Q–Q plot을 기준으로 확인합니다. p≥.05는 가정이 완벽히 성립한다는 증명이 아닙니다. p<.05인 변수의 변환 재분석과 채택 결과는 04-2번 노트북에서 확인합니다.
